In [12]:
from config import *

import time 

import Operators2D as Op

class twoD_Minimal_Model:

    def __init__(self, n, nsm, nsc, m, ax, ay, omega=1, alphat=0, delta=0, meff_sc=1, meff_sm = 1 , mu_sc = 0, mu_sm=0, px=False, py=False,  epsilon=0):

        """
        Initializes a new instance of the oneD_Minimal_Model class.
        
        Args:
            n (int): The number of horizontal lattice points of the system.
            nsm (int): The number of horizontal lattice points of the semiconductor weak link.
            nsc (int): The number of horizontal lattice points of the superconductor on both sides.
            m (int): The number of vertical lattive points
            ax (float): The horizontal (x) lattice spacing.
            ay (float): The vertical (y) lattice spacing.
            meff (float): Effective mass 
            alphat (float): spin orbit coupling constant
            mu_s (float): chemical potential of the superconducting region of SNS
            mu_sm (float): chemical potential of the semiconductingconducting region of SNS
            px (bool, optional): Flag indicating periodicity in x direction. Defaults to False.
            epsilon (float): Kx^4 prefactor to remove higher k states from dipping

        Notes:
            - The ordering of lattice points for kx, kxsq, kxcu(imported for operator class file) is from left to right, bottom to top, similar to x and y axis in 2D g

        """

        self.n = n
        self.nsm = nsm
        self.nsc = nsc
        self.m = m
        self.ax = ax
        self.ay = ay
        self.omega = omega
        self.meff_sc = meff_sc
        self.meff_sm = meff_sm
        self.alphat = alphat
        self.delta = delta
        self.mu_sc = mu_sc
        self.mu_sm  = mu_sm
        #self.dis_mean = dis_mean
        #self.dis_var = dis_var
        self.px = px
        self.py = py
        self.epsilon = epsilon
        self.twodOP = Op.Operators_2d(self.n, self.m, self.ax, self.ay, self.px, self.py)
        self.onedOP =  Op.Operators_2d(1, self.m, 1, self.ay, self.px, self.py)

    def ysq_matrix(self):

        row = np.empty(0)
        col = np.empty(0)
        data = np.empty(0)

        shift = (((self.m//2))*self.ay)

        #print('SHIFT YSQ: ', shift )

        for j in range(self.m):

            for i in range(self.n):
            
                row = np.append(row, j*self.n + i)
                col = np.append(col, j*self.n + i)
                data = np.append(data, ((j)*self.ay-shift)**2)

        M = csr_matrix((data,(row,col)),shape=(self.m*self.n, self.m*self.n))

        return M

    def SHO_wvfn(self, ne, yj=0):

        """
        Return Phi__ne(yj) where yj is the jth index and ne is the quantum number of SHO wavefunction and the whole array
        """

        # define constant of SHo wavefunctions 

        #kappa = self.omega/hbm*self.meff_sm
        kappa = self.omega/hbm*self.meff_sm
        normalization_const = 1/math.sqrt(2**ne * math.factorial(ne))*(kappa/2/math.pi)**(0.25)*self.ay**(0.5)

        # Create the Hermite polynomial 
        coefficients = np.zeros(ne+1)
        coefficients[ne] = 1  # Setting the leading coefficient as per the physicist's definition
        
        H_n = Hermite(coefficients)

        shift = (((self.m//2))*self.ay) ### same shift from as the ysq_matrix 

        #print('SHIFT Y_array: ', shift )

        indices = np.arange(self.m)

        y_array = (indices ) * self.ay  - shift

        y_array_n = math.sqrt(kappa/2)*y_array

        Herm_array = H_n(y_array_n)

        #return normalization_const*np.exp(-kappa/4*y_array**2)*Herm_array, y_array

        wvfn = normalization_const*np.exp(-kappa/4*y_array**2)*Herm_array
        return wvfn[yj], wvfn       

    def SHO_pot(self):

        return 1/4/hbm*self.meff_sm*self.omega**2*self.ysq_matrix() 
    
    def SHO_Hamiltonian_in_y(self,kx=0, ky=0):

        return self.SHO_pot() + hbm/self.meff_sm* self.twodOP.KySq(ky) + hbm/self.meff_sm* self.twodOP.KxSq(kx)

    def SHO_Hamiltonian_in_y_ph(self,ky=0):

        M = bmat([
           [self.SHO_Hamiltonian_in_y(ky), None],
           [None, -self.SHO_Hamiltonian_in_y(-ky).conjugate()]
       ])
        
        return M

    def U_dag(self, my):

        """
        Returns sparse matrix of shape (my * n, m * n)

        Args:
            my: No. of y subbands 
        """
        row = []
        col = []
        data = []        

        for i in range(self.n):
            for j in range(my):
                d, p = self.SHO_wvfn(ne=j, yj=0)
                for k in range(self.m):
                    row.append(j * self.n + i)    
                    col.append(k * self.n + i)               
                    data.append(p[k])

        M = csr_matrix((data, (row, col)), shape=(my * self.n, self.m * self.n))

        return M

    def U_2(self, my, kx=0, ky=0):

        """
        Returns U matrix using eigenvalues found from numerical model 
        """
        

        H = 1/4/hbm*self.meff_sm*self.omega**2*self.onedOP.ysq() + hbm/self.meff_sm* self.onedOP.KySq(ky) 

        E, V = eigsh(H, k=my, sigma=0, return_eigenvectors=True)

        sorted_index = np.argsort(E)

        sorted_V = V[:, sorted_index]

        sorted_E = E[sorted_index]

        #print(sorted_E)

        #print(sorted_V.shape)

        row = []
        col = []
        data = []

        for i in range(self.n):

            for j in range(self.m):

                for k in range(my):

                    row.append(i + self.n*j)    
                    col.append(i + self.n*k)              
                    data.append(sorted_V[j, k])

        M = csr_matrix((data, (row, col)), shape=( self.m * self.n, my * self.n))

        return M

    def U_dag_ph(self, my=1):

        M = bmat([
            [ self.U_dag(my), None],
            [None, -self.U_dag(my).conjugate()],
        ])

        return M
    
    def U_2_ph(self, my=1):

        M = bmat([
            [ self.U_2(my), None],
            [None, self.U_2(my).conjugate()],
        ])

        return M

    def Hamiltonian_particle(self):

        return hbm/self.meff_sm*self.twodOP.KxSq() + hbm/self.meff_sm*self.twodOP.KySq() - self.twodOP.iden()*self.mu_sc + self.SHO_pot()
    
    def Hamiltonian_sys(self):
        
        M = bmat([
           [self.Hamiltonian_particle(), None],
           [None, -self.Hamiltonian_particle().conjugate()]
       ]) 

        return M

In [13]:
NSM = 60            # number of sm x lattice points
M = 50              # should be atleast thrice confinement length
AX = 2.7              # lattice spacing in nm
AY = 2.9              # lattice spacing in nm
#DELTA = 0.165       # in meV
DELTA = 0.00       # in meV
MEFF_SC = 0.023     # proximitized InAs
MEFF_SM = 0.023     # InAs
NSC = 80
NTOT = 2*NSC + NSM
PHI = 0.5*np.pi 
#PHI = 0.0*np.pi
#ALPHAT = 400        # meV-nm^3 # 20 is within perturbative regime for widths atleast up to 100 nm
K_NUM = 4           # No of eigenvalues needed to be found 
#W_MIN = 50
#MY = 30

LY = M*AY//8         # Width in nm

print(M*AY, LY)
OMEGA = 2*4*hbm/LY**2/MEFF_SC

print(OMEGA/2, 3*OMEGA/2)

print('4t=', 4*hbm/AX**2/MEFF_SC)

MU_SC =   OMEGA/2 +OMEGA/10  # in between two subbands - such that we get two abs prferably 
MU_SM =   OMEGA/2 +OMEGA/10  # in between two subbands - such that we get two abs prferably

print(MU_SM)


A = twoD_Minimal_Model(NTOT,NSM,NSC,M,AX,AY,OMEGA, delta=DELTA, meff_sc=MEFF_SC, meff_sm=MEFF_SM, mu_sm = MU_SM, mu_sc=MU_SC)
H = A.Hamiltonian_sys()

E = eigsh(H ,k=K_NUM ,sigma=0, return_eigenvectors=False)

sorted_index = np.argsort(E)

Esorted = E[sorted_index]
#Vsorted = V[:,sorted_index]

print(Esorted)

print(H.shape)

145.0 18.0
20.468272017391353 61.404816052174056
4t= 909.7009785507266
24.561926420869625
[-0.50989524 -0.36063316  0.36063316  0.50989524]
(22000, 22000)


In [14]:
UD = A.U_dag_ph(my=1)
U = UD.conjugate().transpose()
B = UD.dot(H.dot(U))


E = eigsh(B ,k=K_NUM ,sigma=0, return_eigenvectors=False)

sorted_index = np.argsort(E)

Esorted = E[sorted_index]
#Vsorted = V[:,sorted_index]

print(Esorted)

print(H.shape)


[-0.50787425 -0.36265415  0.36265415  0.50787425]
(22000, 22000)


In [15]:
U = A.U_2_ph(my=1)
UD = U.conjugate().transpose()
B = UD.dot(H.dot(U))


E = eigsh(B ,k=K_NUM ,sigma=0, return_eigenvectors=False)

sorted_index = np.argsort(E)

Esorted = E[sorted_index]
#Vsorted = V[:,sorted_index]

print(Esorted)

print(H.shape)

[20.33457017]
(50, 1)
[20.33457017]
(50, 1)
[-0.50989524 -0.36063316  0.36063316  0.50989524]
(22000, 22000)


In [4]:
NSM = 60            # number of sm x lattice points
M = 50              # should be atleast thrice confinement length
AX = 2.7              # lattice spacing in nm
AY = 2.9              # lattice spacing in nm
#DELTA = 0.165       # in meV
DELTA = 0.00       # in meV
MEFF_SC = 0.023     # proximitized InAs
MEFF_SM = 0.023     # InAs
NSC = 80
NTOT = 2*NSC + NSM
PHI = 0.5*np.pi 
#PHI = 0.0*np.pi
#ALPHAT = 400        # meV-nm^3 # 20 is within perturbative regime for widths atleast up to 100 nm
K_NUM = 4           # No of eigenvalues needed to be found 
#W_MIN = 50
#MY = 30

A = twoD_Minimal_Model(NTOT,NSM,NSC,M,AX,AY,OMEGA, delta=DELTA, meff_sc=MEFF_SC, meff_sm=MEFF_SM, mu_sm = MU_SM, mu_sc=MU_SC)

H = A.SHO_Hamiltonian_in_y()

E, V = eigsh(H, k=6, sigma=0, return_eigenvectors=True)

print('E before basis change:', E)

Uop,sV, usV = A.U_2(my=1)

Uopd = Uop.conjugate().transpose() 

H_Changed_basis = Uopd.dot(H.dot(Uop))

E = eigsh(H_Changed_basis, k=6, sigma=1e-5, return_eigenvectors=False)

print('E after numerical basis change one subband:', E)

#print('h changed basis shape:', H_Changed_basis.shape)

UD = A.U_dag(my=1)
U = UD.conjugate().transpose()
B = UD.dot(H.dot(U))


E = eigsh(B ,k=6 ,sigma=0, return_eigenvectors=False)

sorted_index = np.argsort(E)

Esorted = E[sorted_index]
#Vsorted = V[:,sorted_index]

print('E after analytical basis change one subband:', Esorted)

#print(H.shape)


E before basis change: [20.38052662 20.51838671 20.74812258 21.06968779 21.48301737 21.9880278 ]
[20.33457017]
(50, 1)
E after numerical basis change one subband: [20.38052662 20.51838671 20.74812258 21.06968779 21.48301737 21.9880278 ]
E after analytical basis change one subband: [20.38254761 20.5204077  20.75014357 21.07170878 21.48503836 21.99004879]


In [5]:
print(OMEGA)

40.936544034782706


In [6]:
M

50

In [7]:
print(V[:10,:3])

[[-7.58296125e-17  1.52192557e-16 -2.29122128e-16]
 [-1.51447589e-16  3.01875362e-16 -4.55347385e-16]
 [-2.25459506e-16  4.53979124e-16 -6.73679665e-16]
 [-3.02392874e-16  6.00693711e-16 -9.04040753e-16]
 [-3.77159050e-16  7.52511637e-16 -1.12347780e-15]
 [-4.51984768e-16  9.01561775e-16 -1.34125145e-15]
 [-5.28833267e-16  1.04780137e-15 -1.56746156e-15]
 [-6.01470191e-16  1.20008626e-15 -1.77112834e-15]
 [-6.78350191e-16  1.34367700e-15 -1.99119357e-15]
 [-7.52608165e-16  1.48949160e-15 -2.19730624e-15]]


In [76]:
print(sV[:10,:3])

[[ 3.09324039e-17  2.51554920e-15 -2.99035763e-15]
 [-2.03890961e-15  3.82223881e-15  5.30085840e-16]
 [-2.00772320e-15 -4.72791617e-15  2.90028182e-15]
 [-1.82959455e-15  5.75158159e-15 -1.50543417e-15]
 [ 3.32302465e-16 -1.48734653e-15 -8.64835122e-16]
 [ 8.86364595e-16 -1.16254317e-15 -1.68564677e-16]
 [-3.92363285e-16  9.21320865e-16  2.22102337e-15]
 [ 3.03335092e-16  3.42078115e-15 -7.21796771e-18]
 [-1.71873776e-16  2.58713676e-15 -1.29735831e-16]
 [ 2.33584416e-16 -8.20704382e-16  4.24624700e-16]]


In [67]:
print(usV[:10,:3])

[[-7.69659374e-17 -1.51305844e-16 -2.20173797e-16]
 [-1.50088589e-16 -3.01795884e-16 -4.57694223e-16]
 [-2.24847977e-16 -4.52445621e-16 -6.78203752e-16]
 [-3.01736793e-16 -6.02826547e-16 -9.00439047e-16]
 [-3.76986257e-16 -7.52582154e-16 -1.12555512e-15]
 [-4.52048394e-16 -9.01621449e-16 -1.34161914e-15]
 [-5.27674570e-16 -1.05028975e-15 -1.56479044e-15]
 [-6.03409397e-16 -1.19760507e-15 -1.77509802e-15]
 [-6.77843209e-16 -1.34443838e-15 -1.99030763e-15]
 [-7.50954930e-16 -1.48938448e-15 -2.19995588e-15]]
